In [10]:
#imports
import os
import shutil
import glob
import cv2
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
import kagglehub
import tensorflow as tf
from tensorflow.keras import layers, models

In [18]:
# download data
path = kagglehub.dataset_download("fpeccia/weed-detection-in-soybean-crops")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'weed-detection-in-soybean-crops' dataset.
Path to dataset files: /kaggle/input/weed-detection-in-soybean-crops


In [19]:
# function to look at the directory structure to find the image path
def explore_data(base_path):
    for root, dirs, files in os.walk(base_path):
        level = root.replace(base_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/ ({len(files)} files)")

# function to find structural properties of the images
def check_sample_image(img_path):
    img = cv2.imread(img_path)
    if img is not None:
        print(f"\nSample Image Shape: {img.shape}")
        print(f"Pixel Range: {img.min()} to {img.max()}")

        # show a sample image
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title("Sample Dataset Image")
        plt.show()
    else:
        print("Could not read image.")

# apply the functions to the data
explore_data(path)

# find the first image file to inspect
sample_file = None
for root, dirs, files in os.walk(path):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            sample_file = os.path.join(root, f)
            break
    if sample_file: break

if sample_file:
    check_sample_image(sample_file)

weed-detection-in-soybean-crops/ (0 files)
    dataset/ (0 files)
        soil/ (3249 files)
        dataset/ (0 files)
            soil/ (3249 files)
            broadleaf/ (1191 files)
            grass/ (3520 files)
            soybean/ (7376 files)
        broadleaf/ (1191 files)
        grass/ (3520 files)
        soybean/ (7376 files)


In [21]:
# define the root of the cache (currently has a very messy structure)
cache_root = '/root/.cache/kagglehub/datasets'
target_base = '/content/clean_dataset'
classes = ['broadleaf', 'soybean', 'soil', 'grass']

# create target folders
for cls in classes:
    os.makedirs(os.path.join(target_base, cls), exist_ok=True)

print("Searching for images in cache...")

# Search for all image files recursively (was being weird - could not find images in path)
all_images = glob.glob(os.path.join(cache_root, '**/*.tif'), recursive=True) + \
             glob.glob(os.path.join(cache_root, '**/*.jpg'), recursive=True)

copy_count = 0
for img_path in all_images:
    # this determines the class by looking at the folder name in the path
    path_parts = img_path.lower().split(os.sep)
    for cls in classes:
        if cls in path_parts:
            # copy to the clean folder if it finds a match
            filename = os.path.basename(img_path)
            shutil.copy2(img_path, os.path.join(target_base, cls, filename))
            copy_count += 1
            break

print(f" Success! Copied {copy_count} images to {target_base}")


Searching for images in cache...
✅ Success! Copied 30672 images to /content/clean_dataset


In [22]:
for cls in classes:
    count = len(os.listdir(os.path.join(target_base, cls)))
    print(f"{cls}: {count} images")


broadleaf: 1191 images
soybean: 7376 images
soil: 3249 images
grass: 3520 images


In [24]:
# have to convert TIFF files to JPEG so that tensorflow can read them
target_base = '/content/clean_dataset'
classes = ['broadleaf', 'soybean', 'soil', 'grass']

print("Converting TIFFs to JPEGs for TensorFlow...")

for cls in classes:
    cls_path = os.path.join(target_base, cls)
    files = [f for f in os.listdir(cls_path) if f.lower().endswith('.tif')]

    for f in tqdm(files, desc=f"Converting {cls}"):
        img_path = os.path.join(cls_path, f)
        # read the TIFF file
        img = cv2.imread(img_path)
        if img is not None:
            # create a JPEG filename
            new_name = f.rsplit('.', 1)[0] + '.jpg'
            # save image as JPEG and delete TIFF file
            cv2.imwrite(os.path.join(cls_path, new_name), img)
            os.remove(img_path)

print("Conversion complete! TensorFlow can now read the files.")


Converting TIFFs to JPEGs for TensorFlow...


Converting grass: 100%|██████████| 3520/3520 [00:07<00:00, 459.85it/s]

✅ Conversion complete! TensorFlow can now see the files.


In [25]:
# point to the cleaned dataset
base_dir = '/content/clean_dataset'

# create a training set
train_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# create a validation set
val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# adding class weights since there are disproportionate amount of images in each class
# class map: broadleaf:0, grass:1, soil:2, soybean:3 (based on order of files)
class_weight = {0: 3.22, 1: 1.09, 2: 1.18, 3: 0.52}

Found 15336 files belonging to 4 classes.
Using 12269 files for training.
Found 15336 files belonging to 4 classes.
Using 3067 files for validation.


In [27]:
# split the data - reserve 85% for training + validation
train_val_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    validation_split=0.15, # save 15% for the Test Set
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    base_dir,
    validation_split=0.15,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# split 85% into Train (70%) and Val (15%) by calculating the number of batches
batches = tf.data.experimental.cardinality(train_val_ds)
val_batches = batches // 5 # 20% of 85%

val_ds = train_val_ds.take(val_batches)
train_ds = train_val_ds.skip(val_batches)

print(f"Batches - Train: {tf.data.experimental.cardinality(train_ds)}, Val: {tf.data.experimental.cardinality(val_ds)}, Test: {tf.data.experimental.cardinality(test_ds)}")


Found 15336 files belonging to 4 classes.
Using 13036 files for training.
Found 15336 files belonging to 4 classes.
Using 2300 files for validation.
Batches - Train: 327, Val: 81, Test: 72


In [29]:
# using MobileNetV3-Large since im going to deploy model to web app (model is small enough to load quickly)
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# top layer
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    # preprocessing is built in so dont need lambda
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3), # adding a dropout to avoid overfitting
    layers.Dense(4, activation='softmax')
])

# define early stopping rule
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',     # monitor validation loss
    patience=3,             # stop if it doesn't improve after 3 epochs
    restore_best_weights=True # keep best version
)


model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# train the model
model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=class_weight, callbacks=[early_stop])


Epoch 1/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 426s 1s/step - accuracy: 0.9131 - loss: 0.2828 - val_accuracy: 0.9838 - val_loss: 0.0681
Epoch 2/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 429s 1s/step - accuracy: 0.9787 - loss: 0.0808 - val_accuracy: 0.9911 - val_loss: 0.0394
Epoch 3/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 445s 1s/step - accuracy: 0.9839 - loss: 0.0589 - val_accuracy: 0.9942 - val_loss: 0.0299
Epoch 4/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 441s 1s/step - accuracy: 0.9873 - loss: 0.0471 - val_accuracy: 0.9950 - val_loss: 0.0242
Epoch 5/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 442s 1s/step - accuracy: 0.9883 - loss: 0.0388 - val_accuracy: 0.9938 - val_loss: 0.0265
Epoch 6/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 409s 1s/step - accuracy: 0.9905 - loss: 0.0349 - val_accuracy: 0.9942 - val_loss: 0.0227
Epoch 7/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 406s 1s/step - accuracy: 0.9904 - loss: 0.0316 - val_accuracy: 0.9942 - val_loss: 0.0177
Epoch 8/15
327/327 ━━━━━━━━━━━━━━━━━━━━ 449s 1s/step - accuracy: 0.9921 - loss: 0.0277 - val_accu

In [30]:
results = model.evaluate(test_ds)
print(f"Final Unbiased Accuracy on Test Set: {results[1]*100:.2f}%")

72/72 ━━━━━━━━━━━━━━━━━━━━ 73s 1s/step - accuracy: 0.9974 - loss: 0.0105
Final Unbiased Accuracy on Test Set: 99.74%


In [31]:
# save the model to file
model.save('soybean_weed_model.keras')

print("model saved as soybean_weed_model.keras")


model saved as soybean_weed_model.keras
